# Gợi ý bài tập thích ứng bằng truy vết kiến thức

**Sổ Nghiên Cứu AI, dự án mẫu 10/10.** So sánh truy vết kiến thức Bayes (BKT) với mô hình học sâu (DKT) khi dữ liệu nhỏ như ở một trường.

**Câu hỏi nghiên cứu:** BKT có dự đoán câu trả lời kế tiếp của học sinh tốt bằng DKT trên dữ liệu nhỏ không?

**Giả thuyết:** Với dưới 5.000 lượt trả lời, BKT đạt AUC tương đương DKT và dễ giải thích hơn.

### Cách dùng notebook này
1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** nếu có (không bắt buộc).
2. Chạy lần lượt từng ô bằng **Shift + Enter**. Lần đầu có thể mất vài phút để cài thư viện và tải dữ liệu.
3. Các ô **Việc của em** là chỗ em tự làm tiếp để biến dự án mẫu thành nghiên cứu của riêng mình.

Notebook tự mô phỏng học sinh làm bài theo đúng giả định của BKT để em có dữ liệu ngay. Khi có dữ liệu thật, đặt file `lam_bai.csv` (cột `hoc_sinh, ky_nang, dung, thu_tu`).

> **Liêm chính khoa học.** Đây là mã mẫu để luyện tập. Nếu dùng cho dự án dự thi, hãy ghi rõ trong sổ nghiên cứu và báo cáo đoạn mã nào lấy từ notebook này, đoạn nào em tự viết hoặc sửa. Dữ liệu dự thi phải do em thu thập hoặc lấy từ nguồn công khai có ghi nguồn.

In [ ]:
import os, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 110
def tieu_de(s): print("\n" + "=" * 8, s, "=" * 8)

import torch, itertools
from torch import nn
from sklearn.metrics import roc_auc_score
_ = torch.manual_seed(SEED)

## 1. Dữ liệu làm bài

In [ ]:
KY_NANG = ["phan_so", "phuong_trinh_bac_1", "ham_so", "he_phuong_trinh", "ti_le_phan_tram"]
THAT = {k: dict(L0=np.random.uniform(.1, .4), T=np.random.uniform(.08, .25), G=np.random.uniform(.15, .3), S=np.random.uniform(.05, .15)) for k in KY_NANG}
def mo_phong(so_hs=120, moi_ky_nang=12, rng=np.random.default_rng(SEED)):
    rows = []
    for h in range(so_hs):
        thu_tu = 0
        for k in rng.permutation(KY_NANG):
            p = THAT[k]; biet = rng.random() < p["L0"]
            for _ in range(moi_ky_nang):
                dung = rng.random() < (1 - p["S"] if biet else p["G"])
                rows.append((h, k, int(dung), thu_tu)); thu_tu += 1
                if not biet and rng.random() < p["T"]: biet = True
    return pd.DataFrame(rows, columns=["hoc_sinh", "ky_nang", "dung", "thu_tu"])
if os.path.exists("lam_bai.csv"): DL = pd.read_csv("lam_bai.csv"); NGUON = "của em"
else: DL = mo_phong(); NGUON = "GIẢ LẬP theo mô hình BKT"
DL = DL.sort_values(["hoc_sinh", "thu_tu"]).reset_index(drop=True); KY_NANG = sorted(DL.ky_nang.unique())
print(NGUON, "|", DL.hoc_sinh.nunique(), "học sinh |", len(DL), "lượt trả lời |", len(KY_NANG), "kỹ năng")
DL.assign(lan=DL.groupby(["hoc_sinh", "ky_nang"]).cumcount() + 1).groupby("lan").dung.mean().plot(marker="o", title="Đường học tập: tỉ lệ đúng theo lần thử"); plt.xlabel("lần thử thứ"); plt.show()

## 2. Chia theo học sinh
Học sinh trong tập kiểm tra **không xuất hiện** khi huấn luyện, giống như khi dùng hệ thống cho một lớp mới.

In [ ]:
hs = DL.hoc_sinh.unique(); np.random.shuffle(hs); hs_te = set(hs[: len(hs) // 4])
TR, TE = DL[~DL.hoc_sinh.isin(hs_te)], DL[DL.hoc_sinh.isin(hs_te)]
print("Học sinh huấn luyện:", TR.hoc_sinh.nunique(), "| kiểm tra:", TE.hoc_sinh.nunique())

## 3. BKT tự cài đặt
Mỗi kỹ năng có 4 tham số: biết từ đầu (L0), học được sau mỗi bài (T), đoán mò đúng (G), lỡ tay sai (S). Ta tìm bộ tham số làm dữ liệu huấn luyện “hợp lý” nhất bằng tìm kiếm lưới.

In [ ]:
def bkt_du_doan(day, L0, T, G, S):
    L, out = L0, []
    for d in day:
        out.append(L * (1 - S) + (1 - L) * G)
        post = L * (1 - S) / (L * (1 - S) + (1 - L) * G) if d else L * S / (L * S + (1 - L) * (1 - G))
        L = post + (1 - post) * T
    return out
LUOI = list(itertools.product([.05, .15, .3, .5], [.05, .1, .2, .3], [.1, .2, .3], [.05, .1, .2]))
def fit_bkt(df):
    ts = {}
    for k in KY_NANG:
        days = [g.dung.values for _, g in df[df.ky_nang == k].groupby("hoc_sinh")]
        def ll(p): return sum(np.sum(np.log(np.clip(np.where(d, bkt_du_doan(d, *p), 1 - np.array(bkt_du_doan(d, *p))), 1e-6, 1))) for d in days)
        ts[k] = max(LUOI, key=ll)
    return ts
def danh_gia_bkt(ts, df):
    y, p = [], []
    for (h, k), g in df.groupby(["hoc_sinh", "ky_nang"]): y += list(g.dung); p += bkt_du_doan(g.dung.values, *ts[k])
    return roc_auc_score(y, p)
TS = fit_bkt(TR)
print("AUC của BKT trên học sinh mới:", round(danh_gia_bkt(TS, TE), 3))
print(pd.DataFrame(TS, index=["L0 biết sẵn", "T học được", "G đoán mò", "S lỡ tay"]).T.round(2))

## 4. DKT: mạng LSTM đọc toàn bộ lịch sử làm bài

In [ ]:
EPOCHS = int(os.environ.get("SNC_EPOCHS", 40))
K = len(KY_NANG); id_kn = {k: i for i, k in enumerate(KY_NANG)}
def chuoi(df):
    out = []
    for _, g in df.groupby("hoc_sinh"):
        k = g.ky_nang.map(id_kn).values; d = g.dung.values; out.append((k, d))
    return out
class DKT(nn.Module):
    def __init__(s): super().__init__(); s.l = nn.LSTM(2 * K, 64, batch_first=True); s.o = nn.Linear(64, K)
    def forward(s, x): return torch.sigmoid(s.o(s.l(x)[0]))
def to_tensor(seqs):
    L = max(len(k) for k, _ in seqs); X = torch.zeros(len(seqs), L, 2 * K); KN = torch.zeros(len(seqs), L, dtype=torch.long); Y = torch.full((len(seqs), L), -1.0)
    for i, (k, d) in enumerate(seqs):
        X[i, np.arange(len(k)), k + K * d] = 1; KN[i, :len(k)] = torch.tensor(k); Y[i, :len(k)] = torch.tensor(d, dtype=torch.float)
    return X, KN, Y
def danh_gia_dkt(m, seqs):
    X, KN, Y = to_tensor(seqs); m.eval()
    with torch.no_grad(): P = m(X)
    p = P[:, :-1].gather(2, KN[:, 1:, None]).squeeze(-1); y = Y[:, 1:]; mk = y >= 0
    return roc_auc_score(y[mk].numpy(), p[mk].numpy())
def train_dkt(seqs):
    m = DKT(); opt = torch.optim.Adam(m.parameters(), 3e-3); X, KN, Y = to_tensor(seqs)
    for ep in range(EPOCHS):
        m.train(); perm = torch.randperm(len(X))
        for i in range(0, len(X), 8):                                  # lô nhỏ 8 học sinh
            b = perm[i:i + 8]; opt.zero_grad(); P = m(X[b])
            p = P[:, :-1].gather(2, KN[b][:, 1:, None]).squeeze(-1); y = Y[b][:, 1:]; mk = y >= 0
            nn.functional.binary_cross_entropy(p[mk], y[mk]).backward(); opt.step()
    return m
dkt = train_dkt(chuoi(TR))
print("AUC của DKT trên học sinh mới:", round(danh_gia_dkt(dkt, chuoi(TE)), 3))

## 5. Kiểm tra giả thuyết: thay đổi lượng dữ liệu huấn luyện

In [ ]:
KQ = []
hs_tr = TR.hoc_sinh.unique()
for n in [10, 25, 50, len(hs_tr)]:
    for lan in range(3):
        chon = np.random.default_rng(lan).choice(hs_tr, size=min(n, len(hs_tr)), replace=False); sub = TR[TR.hoc_sinh.isin(chon)]
        KQ.append({"so_luot": len(sub), "BKT": danh_gia_bkt(fit_bkt(sub), TE), "DKT": danh_gia_dkt(train_dkt(chuoi(sub)), chuoi(TE))})
R = pd.DataFrame(KQ).groupby("so_luot").agg(["mean", "std"]).round(3); print(R)
for m_ in ["BKT", "DKT"]: plt.errorbar(R.index, R[(m_, "mean")], yerr=R[(m_, "std")], marker="o", capsize=3, label=m_)
plt.xscale("log"); plt.xlabel("số lượt trả lời dùng để huấn luyện"); plt.ylabel("AUC trên học sinh mới"); plt.legend(); plt.show()

## Việc của em
1. Dữ liệu mô phỏng được sinh ra **theo đúng giả định của BKT**, nên BKT có lợi thế. Đây là một hạn chế phải nêu. Hãy thử trên dữ liệu thật: bộ ASSISTments công khai, hoặc dữ liệu lớp thí điểm có đồng ý.
2. Xây bản đồ kỹ năng cho một chương Toán theo chương trình hiện hành, mỗi câu hỏi gắn với một kỹ năng.
3. Thử gợi ý bài: luôn chọn kỹ năng có xác suất “đã biết” thấp nhất. So sánh mức tiến bộ với nhóm làm bài theo thứ tự cố định.